In [2]:
import open3d as o3d
from pathlib import Path

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


1. Mesh result from deskewed cloud point

In [3]:
deskewed_ply_path = Path(r"D:\project2\data\round1\output_mapping_plus_withdeskewed\mesh_fused.ply")

deskewed_mesh = o3d.io.read_triangle_mesh(str(deskewed_ply_path))
deskewed_mesh.compute_vertex_normals()
o3d.visualization.draw_geometries([deskewed_mesh])

2. Mesh result without deskewed cloud point

In [ ]:
without_deskewed_ply_path = Path(r"D:\project2\data\round1\output_mapping_plus\mesh_fused.ply")

without_deskewed_mesh = o3d.io.read_triangle_mesh(str(without_deskewed_ply_path))
without_deskewed_mesh.compute_vertex_normals()
o3d.visualization.draw_geometries([without_deskewed_mesh])

3.vis raw data

In [ ]:
from pathlib import Path
import numpy as np
import open3d as o3d
from rosbags.highlevel import AnyReader

# ROS PointField datatype -> numpy dtype
# INT8=1 UINT8=2 INT16=3 UINT16=4 INT32=5 UINT32=6 FLOAT32=7 FLOAT64=8
_PF_DT = {
    1: np.dtype(np.int8),
    2: np.dtype(np.uint8),
    3: np.dtype(np.int16),
    4: np.dtype(np.uint16),
    5: np.dtype(np.int32),
    6: np.dtype(np.uint32),
    7: np.dtype(np.float32),
    8: np.dtype(np.float64),
}

def pointcloud2_to_xyz_i(msg):
    if msg.point_step <= 0:
        raise ValueError("Invalid point_step")

    n_points = int(msg.width) * int(msg.height)
    if n_points == 0:
        return np.empty((0, 3), dtype=np.float64), None

    names, formats, offsets = [], [], []
    for f in msg.fields:
        if f.datatype not in _PF_DT:
            continue
        base = _PF_DT[f.datatype]
        if int(f.count) <= 1:
            names.append(f.name)
            formats.append(base)
            offsets.append(int(f.offset))
        else:
            for k in range(int(f.count)):
                names.append(f"{f.name}_{k}")
                formats.append(base)
                offsets.append(int(f.offset) + k * base.itemsize)

    dtype = np.dtype(
        {"names": names, "formats": formats, "offsets": offsets, "itemsize": int(msg.point_step)}
    )
    dtype = dtype.newbyteorder(">" if bool(msg.is_bigendian) else "<")

    arr = np.frombuffer(msg.data, dtype=dtype, count=n_points)

    if "x" not in arr.dtype.names or "y" not in arr.dtype.names or "z" not in arr.dtype.names:
        raise ValueError(f"PointCloud2 missing x/y/z, got: {arr.dtype.names}")

    xyz = np.stack([arr["x"], arr["y"], arr["z"]], axis=1).astype(np.float64, copy=False)

    m = np.isfinite(xyz).all(axis=1)
    xyz = xyz[m]

    intensity = None
    if "intensity" in arr.dtype.names:
        intensity = arr["intensity"].astype(np.float64, copy=False)[m]

    return xyz, intensity


def read_all_pointclouds_as_one(
    bag_dir: str,
    topic: str,
    every_n: int = 1,         
    per_frame_voxel: float = 0.0, 
    global_voxel: float = 0.0,   
    max_frames: int = 0,   
):
    bag_path = Path(bag_dir)

    chunks = []
    frame = 0
    taken = 0

    with AnyReader([bag_path]) as reader:
        conns = [c for c in reader.connections if c.topic == topic]
        if not conns:
            raise RuntimeError(f"Topic not found: {topic}")

        for conn, ts, raw in reader.messages(connections=conns):
            if every_n > 1 and (frame % every_n != 0):
                frame += 1
                continue

            msg = reader.deserialize(raw, conn.msgtype)
            xyz, _ = pointcloud2_to_xyz_i(msg)

            if per_frame_voxel and per_frame_voxel > 0 and len(xyz) > 0:
                tmp = o3d.geometry.PointCloud()
                tmp.points = o3d.utility.Vector3dVector(xyz)
                tmp = tmp.voxel_down_sample(float(per_frame_voxel))
                xyz = np.asarray(tmp.points)

            if len(xyz) > 0:
                chunks.append(xyz)

            frame += 1
            taken += 1
            if max_frames and taken >= max_frames:
                break

    if not chunks:
        pcd = o3d.geometry.PointCloud()
        return pcd

    all_xyz = np.concatenate(chunks, axis=0)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(all_xyz)

    if global_voxel and global_voxel > 0:
        pcd = pcd.voxel_down_sample(float(global_voxel))

    return pcd


def visualize_all_as_one(bag_dir: str, topic: str):
    pcd = read_all_pointclouds_as_one(
        bag_dir=bag_dir,
        topic=topic,
        every_n=1,
        per_frame_voxel=0.1, 
        global_voxel=0.0,
        max_frames=0,
    )
    o3d.visualization.draw_geometries([pcd])


def export_all_as_ply(bag_dir: str, topic: str, out_ply: str):
    pcd = read_all_pointclouds_as_one(
        bag_dir=bag_dir,
        topic=topic,
        every_n=1,
        per_frame_voxel=0.1,
        global_voxel=0.0,
        max_frames=0,
    )
    o3d.io.write_point_cloud(out_ply, pcd)




In [9]:
visualize_all_as_one(r"D:\project2\rosbag2_2025_11_21-14_48_34_only_lidar\rosbag2_2025_11_21-14_48_34_only_lidar.db3", "/lidar_points")